# Validación de Audit Tracer con MIMIC-III Clinical Database Demo v1.4

Sección 7.4 del documento final: tres casos de uso sobre datos clínicos reales y anonimizados
(MIMIC-III Demo, PhysioNet, licencia ODbL 1.0, doi:10.13026/C2HM2Q).

Cada afirmación del documento es una **verificación** con resultado `CUMPLE` o `NO CUMPLE` y su evidencia.
Las mismas verificaciones corren como pruebas en `tests/test_validacion_mimic.py`.

**Cómo se ejecuta el sistema (sin atajos):**

* el servidor central (`app.py`) corre en su propio proceso, con base central, cuentas y correo de alertas propios;
* la científica de datos se autentica una vez con **código de dispositivo** y sus scripts de pandas corren en procesos aparte,
  con `import audit_tracer` como única línea añadida;
* el administrador y el auditor usan el dashboard por HTTP.

**Reloj:** las operaciones del analista se ejecutan con `validacion/reloj.py`, que fija la hora del día
(10:00 y 03:10, hora de Colombia) sobre la **fecha real de ejecución**; la línea base del caso 2 usa los cinco días anteriores.

Requisito: `python validacion/descargar_mimic.py` (descarga y verifica los CSV desde PhysioNet).

In [1]:
import json, os, sys
from datetime import datetime
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "validacion" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from validacion.casos_mimic import ValidacionMIMIC, carpeta_mimic, COLOMBIA

DATOS = carpeta_mimic()
assert DATOS is not None, "Faltan los CSV: python validacion/descargar_mimic.py"
FECHA = datetime.now(COLOMBIA).date().isoformat()
EVIDENCIA = Path(os.environ.get("VALIDACION_EVIDENCIA_DIR") or RAIZ / "validacion" / "evidencia" / FECHA)
pd.set_option("display.max_colwidth", 120)

def tabla(verificaciones):
    return pd.DataFrame([{"#": v.id, "Afirmación": v.afirmacion, "Estado": v.estado, "Detalle": v.detalle}
                         for v in verificaciones]).style.hide(axis="index")

val = ValidacionMIMIC(EVIDENCIA, DATOS)
print("Datos:", DATOS)
print("Evidencia:", EVIDENCIA)

Datos: C:\Users\alexd\Universidad\OctavoSemestre\PDG\PDG-Code\validacion\datos\mimic-iii-clinical-database-demo-1.4
Evidencia: C:\Users\alexd\Universidad\OctavoSemestre\PDG\PDG-Code\validacion\evidencia\2026-11-21


## Preparación
Servidor central, cuentas (ADMIN, AUDITOR, CIENTIFICO_DATOS), destinatario de alertas (oficial de cumplimiento) y login por código de dispositivo de la científica de datos.

In [2]:
val.preparar()
prep = json.loads((EVIDENCIA / "preparacion.json").read_text(encoding="utf-8"))
print(json.dumps({k: prep[k] for k in ("dataset", "sha256_datos", "login_dispositivo")}, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in val._info.items()}, indent=2, ensure_ascii=False))

{
  "dataset": "MIMIC-III Clinical Database Demo v1.4 (PhysioNet, ODbL 1.0, doi:10.13026/C2HM2Q)",
  "sha256_datos": {
    "PATIENTS.csv": "f65803a46bcb88127574bfd0b70d78d4123783d05bb411e70d4c9629edd7b703",
    "ADMISSIONS.csv": "487e0b87cc81472cc2eb9ee911b3493c39e15d32e8230cec559907060c009b1c",
    "LABEVENTS.csv": "bca32a7242e739c0cbf1690270db83c10e38a27dbfa915eeb14f5a31a1e898fa"
  },
  "login_dispositivo": {
    "success": true,
    "usuario_id": "2ef29d4b-40d6-49a0-a9d2-84a0b4253f78",
    "origen": "device_flow",
    "codigo_mostrado": "CTE5-HDNZ",
    "confirmado_en_dashboard": true
  }
}
{
  "fecha_ejecucion": "2026-11-21T13:23:22-05:00",
  "python": "3.12.6",
  "pandas": "3.0.2",
  "sistema": "Windows 11 (10.0.26200)",
  "maquina": "AMD64",
  "procesador": "AMD64 Family 23 Model 160 Stepping 0, AuthenticAMD",
  "cpus_logicas": "8",
  "commit": "e4a656bdfc928a5b94d073d5a908ac92e31c0506",
  "rama": "entregables-documento-final",
  "fecha_commit": "2026-11-21 13:22:59 -0500",
  "ca

## Caso 1 — Flujo de análisis normal
La científica carga `PATIENTS`, `ADMISSIONS` y `LABEVENTS`, consulta columnas, limpia (fillna, merge, drop, dropna, rename) y exporta el reporte de la cohorte y un anexo de creatinina.

In [3]:
print((RAIZ / "validacion/scripts/caso1_analisis_cohorte.py").read_text(encoding="utf-8"))

"""
Caso de uso 1 — Flujo de análisis normal sobre MIMIC-III Clinical Database Demo v1.4.

Es el código que escribiría un científico de datos para preparar un reporte
clínico de la cohorte de admisiones. La única línea relacionada con la
auditoría es `import audit_tracer`: no hay llamadas a la librería, ni
envoltorios, ni nombres de dataset puestos a mano.

Uso: python caso1_analisis_cohorte.py <carpeta_mimic> <carpeta_salida>
"""
import audit_tracer  # noqa: F401  (única línea añadida al script del analista)

import os
import sys

import pandas as pd

datos, salida = sys.argv[1], sys.argv[2]
os.makedirs(salida, exist_ok=True)

# 1. Carga
pacientes = pd.read_csv(os.path.join(datos, "PATIENTS.csv"))
admisiones = pd.read_csv(os.path.join(datos, "ADMISSIONS.csv"))

# 2. Consulta de las columnas del estudio
admisiones = admisiones[[
    "subject_id", "hadm_id", "admittime", "dischtime", "deathtime",
    "admission_type", "insurance", "language", "diagnosis", "hospital_expire_flag",
]]

# 3

In [4]:
tabla(val.caso1())

[AuditTracer] La auditoría está activa, pero no hay un usuario identificado: los eventos se registran como DESCONOCIDO (nivel CRITICO). Ejecuta audit_tracer.login(api_url=...) para identificarte.


#,Afirmación,Estado,Detalle
1.1,"La carga, la transformación y la exportación se capturan automáticamente sin modificar el código del analista.",CUMPLE,"El script solo contiene `import audit_tracer` (1 import, 0 llamadas a la librería). Eventos en el servidor: CARGA=3 (ADMISSIONS.csv, LABEVENTS.csv, PATIENTS.csv), CONSULTA=2, TRANSFORMACION=6 (ELIMINACION_COLS, ELIMINACION_FILAS, IMPUTACION, MERGE, RENOMBRE), EXPORTACION=2."
1.2,Cada evento queda con la identidad correcta del usuario y el identificador de sesión.,CUMPLE,"15 eventos, todos con usuario_id=2ef29d4b-40d6-49a0-a9d2-84a0b4253f78 (cientifica@…, identificada con el token cacheado del login por dispositivo, sin escribir credenciales) y sesion_id=2ed6163e-069e-4ee2-8d7d-8c25abc4bc70 (de INICIO_SESION a CIERRE_SESION). Los mismos 15 evento_uuid están en la base local y en la central; 15 con firma Ed25519."
1.3,La exportación activa el nivel de alerta según el número de registros exportados.,CUMPLE,reporte_clinico_cohorte.csv: 129 filas → NORMAL; anexo_creatinina.csv: 2174 filas → CRITICO (Exportación masiva: 2174 filas (> 1000)). Umbral del código: > 1000 filas = CRITICO.
1.4a,El grafo de linaje reconstruye la cadena de una tabla sin uniones (LABEVENTS: carga → limpieza → exportación).,CUMPLE,Cadena LABEVENTS.csv en la sesión: CARGA → TRANSFORMACION → EXPORTACION → archivo anexo_creatinina.csv.
1.4b,El grafo de linaje reconstruye la cadena completa de la cohorte (carga de ADMISSIONS → merge con PATIENTS → limpieza → exportación del reporte).,NO CUMPLE,"El grafo se arma por (sesión, dataset_nombre) y el merge le pone otro nombre al resultado, así que la cadena se parte en dos grafos sin arista entre ellos: ADMISSIONS.csv = CARGA → TRANSFORMACION → TRANSFORMACION; merge(ADMISSIONS.csv,PATIENTS.csv) = TRANSFORMACION → TRANSFORMACION → TRANSFORMACION → EXPORTACION → archivo reporte_clinico_cohorte.csv. La carga de PATIENTS.csv queda sola."
1.5,El reporte incluye las referencias normativas correctas por tipo de evento.,CUMPLE,15 filas de la sesión en el CSV del dashboard; todas coinciden con la tabla MAPEO_NORMATIVO (derivada de la matriz HU-6.6). Por tipo: INICIO_SESION = HIPAA §164.312(b) | Ley 1581 Art. 17 lit. m | Matriz CT-17 ; CARGA = HIPAA §164.312(b) | Ley 1581 Art. 17 lit. m | Matriz CT-12 ; CONSULTA = HIPAA §164.312(b) | Ley 1581 Art. 17 lit. m | Matriz CT-12 ; TRANSFORMACION = HIPAA §164.312(b); §164.312(c)(1) | Ley 1581 Art. 4 lit. d; Art. 17 lit. m | Matriz CT-13 ; EXPORTACION = HIPAA §164.312(b); §164.308(a)(1)(ii)(D); §164.528 (apoyo) | Ley 1581 Art. 4 lit. f; Art. 17 lit. m | Matriz CT-14 ; CIERRE_SESION = HIPAA §164.312(b) | Ley 1581 Art. 17 lit. m | Matriz CT-17


In [5]:
eventos = pd.DataFrame(val._caso1["eventos"])
eventos["subtipo"] = eventos["motivo_fallo"].str.extract(r"\[(\w+)\]")
eventos[["event_id", "timestamp", "tipo_accion", "subtipo", "dataset_nombre", "filas_exportadas", "nivel_alerta", "motivo_alerta"]]

,event_id,timestamp,tipo_accion,subtipo,dataset_nombre,filas_exportadas,nivel_alerta,motivo_alerta
0,14,2026-11-21T15:00:00.063000,INICIO_SESION,NaN,NaN,NaN,NORMAL,NaN
1,15,2026-11-21T15:00:00.282000,CARGA,NaN,PATIENTS.csv,NaN,NORMAL,NaN
2,16,2026-11-21T15:00:00.297000,CARGA,NaN,ADMISSIONS.csv,NaN,NORMAL,NaN
3,17,2026-11-21T15:00:00.297000,CONSULTA,NaN,ADMISSIONS.csv,NaN,NORMAL,NaN
4,18,2026-11-21T15:00:00.329000,TRANSFORMACION,IMPUTACION,ADMISSIONS.csv,NaN,NORMAL,NaN
5,19,2026-11-21T15:00:00.344000,TRANSFORMACION,MERGE,ADMISSIONS.csv,NaN,NORMAL,NaN
6,20,2026-11-21T15:00:00.344000,TRANSFORMACION,ELIMINACION_COLS,"merge(ADMISSIONS.csv,PATIENTS.csv)",NaN,NORMAL,NaN
7,21,2026-11-21T15:00:00.360000,TRANSFORMACION,ELIMINACION_FILAS,"merge(ADMISSIONS.csv,PATIENTS.csv)",NaN,NORMAL,NaN
8,22,2026-11-21T15:00:00.360000,TRANSFORMACION,RENOMBRE,"merge(ADMISSIONS.csv,PATIENTS.csv)",NaN,NORMAL,NaN
9,23,2026-11-21T15:00:00.375000,EXPORTACION,NaN,"merge(ADMISSIONS.csv,PATIENTS.csv)",129.0,NORMAL,NaN


## Caso 2 — Comportamiento anómalo
Cinco días de trabajo habitual (muestras de 120 a 180 filas en horario laboral) y luego, a las 03:10, la exportación completa de `LABEVENTS` (76 074 filas).

In [6]:
tabla(val.caso2())

#,Afirmación,Estado,Detalle
2.1,Se activan las dos reglas de anomalía (horario inusual y volumen anómalo) sobre la exportación registrada en el servidor.,NO CUMPLE,"Evento #53 en el servidor: nivel=CRITICO, motivo=""Exportación masiva: 76074 filas (> 1000)"". El detector no se ejecutó: app.py solo lo aplica a eventos SIN firma y que no lleguen como CRITICO, y la librería firma todos los eventos y marca CRITICO toda exportación de más de 1000 filas."
2.1b,"El motor de detección, aplicado al evento con el historial que habría al recibirlo, activa las dos reglas.",NO CUMPLE,"Volumen: Volumen anómalo: 76,074 filas supera significativamente el promedio histórico del usuario (150.0 filas). Horario: no se activa (horario habitual calculado = (7, 19) UTC, que ya incluye el INICIO_SESION y la CARGA de las 03:10 de la misma sesión)."
2.1c,"El motor de detección, con el historial previo a la sesión, activa las dos reglas.",CUMPLE,"Volumen: Volumen anómalo: 76,074 filas supera significativamente el promedio histórico del usuario (150.0 filas). Horario: Horario inusual: acceso registrado a las 08:10 fuera del horario habitual del usuario (13:00 - 19:00) (horario habitual = (13, 19) UTC)."
2.2,El evento queda con su nivel de alerta.,CUMPLE,nivel_alerta=CRITICO (asignado por la librería por superar 1000 filas; no es ANOMALIA porque el detector no corrió).
2.3,Se genera la descripción del motivo de la alerta con las reglas de anomalía.,NO CUMPLE,"motivo_alerta en el servidor: ""Exportación masiva: 76074 filas (> 1000)"". Solo describe la exportación masiva; las descripciones de las reglas de anomalía no llegan al registro (ver 2.1)."
2.4a,Se envía automáticamente la notificación por correo al oficial de cumplimiento.,CUMPLE,"alertas_notificaciones: estado=ENVIADA, tipos=EXPORTACION_MASIVA, severidad=ALTA, intentos=1, destinatarios del correo (To/Cc/Bcc)=audit-tracer@mimic.validacion, admin@mimic.validacion, auditor@mimic.validacion, oficial.cumplimiento@mimic.validacion; asunto «[Audit Tracer][ALTA] Exportación masiva — usuario 2ef29d4b-40d6-49a0-a9d2-84a0b4253f78». El correo lo dispara la regla EXPORTACION_MASIVA (> 1000 filas), no una anomalía. Entrega por la carpeta outbox (AUDIT_ALERTAS_OUTBOX), el modo sin SMTP del sistema; no se probó contra un servidor SMTP real."
2.4b,El envío del correo es independiente del registro del evento (el evento se guarda aunque el correo falle o tarde).,CUMPLE,"Con un servicio de correo que tarda 2 s y falla siempre, insert_event devolvió en 0.009 s, el evento quedó en audit_log y la notificación terminó en FALLIDA tras 3 intentos, en otro hilo."


In [7]:
print(json.dumps(json.loads((EVIDENCIA / "caso2/evaluacion_detector_anomalias.json").read_text(encoding="utf-8")),
                 indent=2, ensure_ascii=False)[:4000])

{
  "al_llegar_el_evento": {
    "eventos_en_historial": 52,
    "promedios_historicos_usuario": {
      "usuario_id": "2ef29d4b-40d6-49a0-a9d2-84a0b4253f78",
      "dias_activos": 5,
      "total_operaciones": 10,
      "total_accesos": 5,
      "total_transformaciones": 0,
      "total_exportaciones": 5,
      "promedio_accesos_diarios": 1.0,
      "promedio_transformaciones_diarias": 0.0,
      "promedio_exportaciones_diarias": 1.0,
      "promedio_total_diario": 2.0,
      "promedio_filas_exportadas": 150.0
    },
    "horario_habitual_utc": [
      7,
      19
    ],
    "regla_volumen": [
      true,
      "Volumen anómalo: 76,074 filas supera significativamente el promedio histórico del usuario (150.0 filas)"
    ],
    "regla_horario": [
      false,
      null
    ],
    "es_anomalo": true,
    "motivos": [
      "Volumen anómalo: 76,074 filas supera significativamente el promedio histórico del usuario (150.0 filas)"
    ],
    "nivel_que_asignaria": "CRITICO",
    "motivo_que

## Caso 3 — Evidencia para auditoría regulatoria
El auditor filtra por fecha, usuario y tipo de operación, descarga los CSV, verifica el hash de un evento y la prueba de Merkle de su lote.

In [8]:
tabla(val.caso3())

#,Afirmación,Estado,Detalle
3.1,"Funcionan los filtros por fecha, usuario y tipo de operación (solos y combinados).",CUMPLE,Período 2026-11-16–2026-11-21: 56 eventos; solo 2026-11-20: 4 (todos con fecha 2026-11-20); usuario: 45; usuario+CARGA: 9; usuario+EXPORTACION: 8. Cada conjunto es idéntico al que devuelve una consulta SQL directa a la base central con los mismos criterios.
3.2a,"El reporte incluye quién, qué, cuándo, sobre qué dataset y la base normativa (HIPAA §164.312(b) y Ley 1581).",CUMPLE,"Columnas del CSV: event_id, usuario_id, timestamp, tipo_accion, dataset_nombre, columnas_afectadas, nivel_alerta, referencia_normativa."
3.2b,"El reporte incluye también la sesión, el origen de la solicitud y la huella de cada evento.",NO CUMPLE,"El CSV no trae: sesion_id (sesión); contexto_ejecucion (origen de la solicitud (script, entorno, host)); hash_integridad (huella para verificar el registro); ruta_destino (destino de la exportación); filas_exportadas (volumen exportado). Esos datos existen en audit_log y se ven en el detalle de cada evento del dashboard, pero no salen en el reporte exportado."
3.3,El CSV exportado es válido aunque los filtros no den resultados.,CUMPLE,Filtro sin coincidencias (auditor + EXPORTACION): respuesta con solo la cabecera (8 columnas) y 0 filas; csv.reader la lee sin errores.
3.4a,La integridad de un evento se puede verificar por su hash.,CUMPLE,Evento #53: SHA-256 recalculado sobre los 13 campos = almacenado (1dc8b6815cbffa7b…). Si se cambia filas_exportadas el hash cambia. El botón «Verificar integridad» del dashboard (POST /verificar-integridad) reporta todos los registros íntegros.
3.4b,"Para lotes históricos, la integridad se verifica con la prueba de Merkle y el ancla publicada.",CUMPLE,"Lote 2026-11-21T08 (hora UTC): publicado por el auditor (POST /merkle/publicar), prueba de inclusión del evento #53 con 2 pasos = válida; raíz publicada = raíz recalculada (135d0aa59496630c…), firma Ed25519 del ancla válida. El ancla es un archivo JSON en la carpeta del servidor (AUDIT_MERKLE_ANCHORS_DIR), no un servicio externo."


In [9]:
pd.read_csv(EVIDENCIA / "caso3/reporte_exportaciones_usuario_periodo.csv")

[AuditTracer] La auditoría está activa, pero no hay un usuario identificado: los eventos se registran como DESCONOCIDO (nivel CRITICO). Ejecuta audit_tracer.login(api_url=...) para identificarte.


,event_id,usuario_id,timestamp,tipo_accion,dataset_nombre,columnas_afectadas,nivel_alerta,referencia_normativa
0,33,2ef29d4b-40d6-49a0-a9d2-84a0b4253f78,2026-11-16 09:15:00,EXPORTACION,LABEVENTS.csv,NaN,NORMAL,HIPAA §164.312(b); §164.308(a)(1)(ii)(D); §164.528 (apoyo) | Ley 1581 Art. 4 lit. f; Art. 17 lit. m | Matriz CT-14
1,37,2ef29d4b-40d6-49a0-a9d2-84a0b4253f78,2026-11-17 10:25:00,EXPORTACION,LABEVENTS.csv,NaN,NORMAL,HIPAA §164.312(b); §164.308(a)(1)(ii)(D); §164.528 (apoyo) | Ley 1581 Art. 4 lit. f; Art. 17 lit. m | Matriz CT-14
2,41,2ef29d4b-40d6-49a0-a9d2-84a0b4253f78,2026-11-18 11:35:00,EXPORTACION,LABEVENTS.csv,NaN,NORMAL,HIPAA §164.312(b); §164.308(a)(1)(ii)(D); §164.528 (apoyo) | Ley 1581 Art. 4 lit. f; Art. 17 lit. m | Matriz CT-14
3,45,2ef29d4b-40d6-49a0-a9d2-84a0b4253f78,2026-11-19 09:45:00,EXPORTACION,LABEVENTS.csv,NaN,NORMAL,HIPAA §164.312(b); §164.308(a)(1)(ii)(D); §164.528 (apoyo) | Ley 1581 Art. 4 lit. f; Art. 17 lit. m | Matriz CT-14
4,49,2ef29d4b-40d6-49a0-a9d2-84a0b4253f78,2026-11-20 10:55:00,EXPORTACION,LABEVENTS.csv,NaN,NORMAL,HIPAA §164.312(b); §164.308(a)(1)(ii)(D); §164.528 (apoyo) | Ley 1581 Art. 4 lit. f; Art. 17 lit. m | Matriz CT-14
5,53,2ef29d4b-40d6-49a0-a9d2-84a0b4253f78,2026-11-21 03:10:01,EXPORTACION,LABEVENTS.csv,NaN,CRITICO,HIPAA §164.312(b); §164.308(a)(1)(ii)(D); §164.528 (apoyo) | Ley 1581 Art. 4 lit. f; Art. 17 lit. m | Matriz CT-14
6,23,2ef29d4b-40d6-49a0-a9d2-84a0b4253f78,2026-11-21 10:00:00,EXPORTACION,"merge(ADMISSIONS.csv,PATIENTS.csv)",NaN,NORMAL,HIPAA §164.312(b); §164.308(a)(1)(ii)(D); §164.528 (apoyo) | Ley 1581 Art. 4 lit. f; Art. 17 lit. m | Matriz CT-14
7,27,2ef29d4b-40d6-49a0-a9d2-84a0b4253f78,2026-11-21 10:00:00,EXPORTACION,LABEVENTS.csv,NaN,CRITICO,HIPAA §164.312(b); §164.308(a)(1)(ii)(D); §164.528 (apoyo) | Ley 1581 Art. 4 lit. f; Art. 17 lit. m | Matriz CT-14


## Resultado

In [10]:
val.cerrar()
resumen = val.finalizar()
print(f"Fecha de ejecución: {resumen['entorno']['fecha_ejecucion']}  ·  commit {resumen['entorno']['commit'][:10]}"
      f"  ·  {resumen['entorno']['duracion_segundos']} s")
print(resumen["totales"])
pd.DataFrame([{"#": v["id"], "Caso": v["caso"].split(" — ")[0], "Estado": v["estado"], "Afirmación": v["afirmacion"]}
              for v in resumen["verificaciones"]]).style.hide(axis="index")

Fecha de ejecución: 2026-11-21T13:23:22-05:00  ·  commit e4a656bdfc  ·  56.8 s
{'cumple': 14, 'no_cumple': 5}


#,Caso,Estado,Afirmación
1.1,Caso 1,CUMPLE,"La carga, la transformación y la exportación se capturan automáticamente sin modificar el código del analista."
1.2,Caso 1,CUMPLE,Cada evento queda con la identidad correcta del usuario y el identificador de sesión.
1.3,Caso 1,CUMPLE,La exportación activa el nivel de alerta según el número de registros exportados.
1.4a,Caso 1,CUMPLE,El grafo de linaje reconstruye la cadena de una tabla sin uniones (LABEVENTS: carga → limpieza → exportación).
1.4b,Caso 1,NO CUMPLE,El grafo de linaje reconstruye la cadena completa de la cohorte (carga de ADMISSIONS → merge con PATIENTS → limpieza → exportación del reporte).
1.5,Caso 1,CUMPLE,El reporte incluye las referencias normativas correctas por tipo de evento.
2.1,Caso 2,NO CUMPLE,Se activan las dos reglas de anomalía (horario inusual y volumen anómalo) sobre la exportación registrada en el servidor.
2.1b,Caso 2,NO CUMPLE,"El motor de detección, aplicado al evento con el historial que habría al recibirlo, activa las dos reglas."
2.1c,Caso 2,CUMPLE,"El motor de detección, con el historial previo a la sesión, activa las dos reglas."
2.2,Caso 2,CUMPLE,El evento queda con su nivel de alerta.
